# Детальный разбор кода проекта "Предсказание цен на автомобили"

## БЛОК 1: Загрузка данных и дообучение ViT

### 1.1. Инициализация и настройки
```python
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
```
**Что делает**: Устанавливает случайное зерно для воспроизводимости результатов. Все случайные процессы (инициализация весов, перемешивание данных) будут одинаковыми при каждом запуске.

### 1.2. Вспомогательные функции загрузки

**`get_direct_file_link()`**:
```python
def get_direct_file_link(mailru_file_url: str) -> str:
    resp = requests.get(mailru_file_url)
    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', page)
```
**Что делает**: 
- Отправляет HTTP-запрос к странице облачного хранилища Mail.ru
- Использует регулярное выражение для извлечения прямой CDN-ссылки из HTML-кода
- Возвращает прямую ссылку для скачивания файла

**`download_from_mailru()`**:
```python
os.system(f"wget --content-disposition '{direct}' -O '{local_name}'")
```
**Что делает**: Использует системную команду wget для скачивания файла по прямой ссылке.

**`build_id_to_paths()`**:
```python
for fname in os.listdir(img_dir):
    if fname.endswith(".jpg"):
        iid = int(fname.split("_")[0])
        id_to_files[iid].append(os.path.join(img_dir, fname))
```
**Что делает**: 
- Проходит по всем файлам в директории с изображениями
- Извлекает ID автомобиля из имени файла (формат: `12345_image1.jpg`)
- Создает словарь, где ключ - ID автомобиля, значение - список путей ко всем его изображениям

### 1.3. Модель FineTunedViT

**Инициализация backbone**:
```python
self.backbone = timm.create_model(
    model_name,
    pretrained=pretrained,
    num_classes=0  # Важно: убираем классификационную голову
)
```
**Что делает**: 
- Загружает предобученную Vision Transformer из библиотеки timm
- `num_classes=0` означает, что мы убираем последний классификационный слой, оставляя только эмбеддинги

**Регрессионная голова**:
```python
self.regression_head = nn.Sequential(
    nn.Dropout(dropout),
    nn.Linear(self.embed_dim, 512),
    nn.GELU(),  # Активационная функция
    nn.Dropout(dropout),
    nn.Linear(512, 128),
    nn.GELU(),
    nn.Linear(128, 1)  # Выход - одно число (цена)
)
```
**Что делает**: 
- Создает последовательность слоев для регрессии
- Dropout слои предотвращают переобучение
- GELU - современная активационная функция, похожая на ReLU но более гладкая
- Последний слой выдает одно число - предсказанную цену

### 1.4. Датасет CarPriceDataset

**`__getitem__` метод**:
```python
def __getitem__(self, idx):
    item_id = self.ids[idx]
    paths = self.id_to_paths.get(item_id, [])[:self.max_images]
    
    if not paths:
        # Создаем нулевое изображение если нет фото
        img = np.zeros((224, 224, 3), dtype=np.uint8)
        return img, self.id_to_price[item_id], 0  # Флаг 0 = нет изображения
    
    # Случайный выбор одного изображения из доступных
    random_path = np.random.choice(paths)
    # Обработка изображения через OpenCV и PIL
    img = cv2.imread(random_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)
    
    if self.transform:
        img = self.transform(img)
        
    return img, self.id_to_price[item_id], 1  # Флаг 1 = есть изображение
```
**Что делает**:
- Для каждого ID автомобиля берет до `max_images` изображений
- Если изображений нет - создает черное изображение и устанавливает флаг 0
- Случайно выбирает одно изображение для обучения (аугментация)
- Преобразует BGR в RGB (OpenCV использует BGR по умолчанию)
- Применяет трансформации для ViT
- Возвращает флаг, чтобы можно было игнорировать примеры без изображений в функции потерь

### 1.5. Процесс обучения

**Трансформации для ViT**:
```python
config = resolve_data_config({}, model=model.backbone)
train_transform = create_transform(**config)
```
**Что делает**: Автоматически создает правильные трансформации изображений для конкретной модели ViT (нормализация, resize, аугментации).

**Цикл обучения**:
```python
for batch_imgs, batch_prices, batch_flags in progress_bar:
    valid_mask = batch_flags == 1
    if valid_mask.sum() > 0:
        loss = criterion(pred_prices[valid_mask], batch_prices[valid_mask])
```
**Что делает**: 
- Вычисляет loss только для тех примеров, где есть реальные изображения (флаг = 1)
- Игнорирует примеры с искусственно созданными нулевыми изображениями

**Сохранение лучшей модели**:
```python
if avg_loss < best_loss:
    best_loss = avg_loss
    torch.save(model.state_dict(), 'fine_tuned_vit.pth')
```
**Что делает**: Сохраняет веса модели только если текущая эпоха показала лучший результат.

## БЛОК 2: Извлечение улучшенных эмбеддингов

### 2.1. Загрузка дообученной модели
```python
model = FineTunedViT()
model.load_state_dict(torch.load('fine_tuned_vit.pth', map_location=DEVICE))
model.eval()  # Режим инференса
```
**Что делает**: 
- Создает новую модель того же архитектурного типа
- Загружает веса из дообученной модели
- Устанавливает модель в режим оценки (отключает dropout, batch norm использует статистику)

### 2.2. Извлечение эмбеддингов

**Процесс для каждого автомобиля**:
```python
for item_id in tqdm(id_list, desc=desc):
    paths = id_to_paths.get(item_id, [])
    if not paths:
        # Нулевой эмбеддинг если нет изображений
        all_embeddings[item_id] = np.zeros(emb_size, dtype=np.float32)
        continue
    
    # Создаем даталоадер для ВСЕХ изображений автомобиля
    img_dataset = ImagePathDataset(paths, transform=transform)
    img_loader = DataLoader(img_dataset, batch_size=64, shuffle=False)
    
    # Пропускаем все изображения через модель
    with torch.no_grad():  # Отключаем вычисление градиентов
        for batch in img_loader:
            batch = batch.to(DEVICE)
            feats = model.get_embeddings(batch)  # Получаем эмбеддинги
            embeddings.append(feats.cpu().numpy())
    
    # Усредняем эмбеддинги по всем изображениям автомобиля
    mean_emb = np.mean(np.vstack(embeddings), axis=0)
    all_embeddings[item_id] = mean_emb.astype(np.float32)
```
**Что делает**:
- Для каждого автомобиля загружает ВСЕ его изображения (не одно случайное как при обучении)
- Пропускает все изображения через backbone модели (без регрессионной головы)
- Усредняет полученные эмбеддинги по всем изображениям автомобиля
- Создает один усредненный эмбеддинг на автомобиль

**Преимущества подхода**:
- Усреднение уменьшает шум от отдельных изображений
- Учитывает все доступные ракурсы автомобиля
- Создает более стабильные и информативные признаки

### 2.3. Сохранение эмбеддингов
```python
train_emb_df = pd.DataFrame.from_dict(train_embeddings, orient='index')
train_emb_df.index.name = 'ID'
train_emb_df.columns = [f'img_emb_{i}' for i in range(EMB_SIZE)]
```
**Что делает**: Преобразует словарь эмбеддингов в DataFrame где:
- Индекс - ID автомобиля
- Колонки - img_emb_0, img_emb_1, ... img_emb_191 (192 признака)
- Сохраняет в parquet для эффективного хранения

## БЛОК 3: Обучение CatBoost на улучшенных эмбеддингах

### 3.1. Предобработка данных

**Логарифмирование пробега**:
```python
train_processed['mileage_log'] = np.log1p(train_processed['mileage'])
```
**Что делает**: 
- `log1p(x) = log(1 + x)` стабилизирует распределение пробега
- Уменьшает влияние экстремальных значений
- Делает данные более нормально распределенными

**Обработка мультивыборных полей**:
```python
def count_options(x):
    if x is None:
        return 0
    if isinstance(x, (list, np.ndarray)):
        valid_count = sum(1 for item in x if item is not None)
        return valid_count
    return 0
```
**Что делает**: Преобразует поля типа "список опций" в числовой признак - количество выбранных опций.

**Объединение данных**:
```python
train_final = train_processed.set_index("ID").join(train_emb, how='left')
```
**Что делает**: Объединяет табличные данные с эмбеддингами изображений по ID автомобиля.

### 3.2. Подготовка признаков для CatBoost

**Обработка категориальных признаков**:
```python
categorical_features = [
    'equipment', 'body_type', 'drive_type', 'engine_type', 'doors_number',
    'color', 'pts', 'steering_wheel', ...
]
```
**Что делает**: Определяет какие признаки CatBoost должен рассматривать как категориальные.

**Автоматическое обнаружение строковых колонок**:
```python
object_columns = X_train.select_dtypes(include=['object']).columns.tolist()
missing_categorical = [col for col in object_columns if col not in categorical_features]
```
**Что делает**: Находит все строковые колонки которые не были указаны вручную и добавляет их в категориальные.

**Обработка числовых признаков**:
```python
X_train[col] = pd.to_numeric(X_train[col].astype(str).str.replace(r'[^\d.-]', '', regex=True), errors='coerce')
```
**Что делает**: 
- Преобразует строки в числа, удаляя все не-цифровые символы
- Например: "150 л.с." → 150

### 3.3. Обучение CatBoost

**Создание Pool объектов**:
```python
train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
valid_pool = Pool(X_val, y_val, cat_features=categorical_features)
```
**Что делает**: Создает специальные объекты Pool которые эффективно хранят данные для CatBoost и указывают какие признаки категориальные.

**Параметры модели**:
```python
model = CatBoostRegressor(
    iterations=4000,           # Максимальное число деревьев
    learning_rate=0.05,        # Скорость обучения
    depth=10,                  # Глубина деревьев
    loss_function='RMSE',      # Функция потерь
    eval_metric='RMSE',        # Метрика для валидации
    task_type='GPU',           # Использование GPU для ускорения
    od_type='Iter',            # Тип ранней остановки
    od_wait=100,               # Ждать 100 итераций без улучшения
    verbose=100                # Вывод каждые 100 итераций
)
```
**Что делает**: Настраивает мощный регрессор с GPU-ускорением и ранней остановкой.

**Обучение с валидацией**:
```python
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
```
**Что делает**: 
- Обучает модель на тренировочных данных
- Мониторит качество на валидационной выборке
- Автоматически выбирает лучшую итерацию based on validation score

### 3.4. Предсказание и сохранение

**Обратное преобразование предсказаний**:
```python
y_pred_log = model.predict(test_pool)  # Предсказания в логарифмической шкале
y_pred = np.expm1(y_pred_log)          # Возвращаем к исходной шкале
```
**Что делает**: 
- `expm1(x) = exp(x) - 1` - обратное преобразование к log1p
- Возвращает предсказания в оригинальной шкале цен

## Ключевые особенности архитектуры

### 1. Двухэтапное обучение
- **Этап 1**: Дообучение ViT на регрессию цен по изображениям
- **Этап 2**: Использование эмбеддингов из дообученной ViT как features для CatBoost

### 2. Обработка отсутствующих изображений
- Нулевые эмбеддинги для автомобилей без фото
- Игнорирование в loss function при обучении ViT

### 3. Усреднение эмбеддингов
- Учет всех доступных изображений автомобиля
- Более стабильные и информативные признаки

### 4. Комбинация модальностей
- Табличные данные: технические характеристики, опции
- Визуальные признаки: эмбеддинги из изображений
- Категориальные и числовые признаки

Такой подход позволяет модели учитывать как объективные характеристики автомобиля, так и его визуальное состояние и внешний вид, что значительно улучшает качество предсказаний.